<br>

<img src="https://lindas.admin.ch/lindaslogo2.png" style="width:25%; float:right">

# LINDAS Tutorial

This page serves as a introductory tutorial to the LINDAS ecosystem. LINDAS is the Linked Data ecosystem of the Swiss Federal Administration.



# Introduction

The webpage you are currently viewing is a so called **interactive JupyterLite notebook**. In this notebook, you can change interactively the content of the single cells and execute these cells directly seeing the result of your changes immediately. The cells contain either [Markdown](https://en.wikipedia.org/wiki/Markdown) content (like this cell) or executable Python source code.

JupyterLite stems from JupyterLab with the advantage of being completely browser based without any backend infrastructure. This means that the execution of the cells could take some time during first execution. Subsequent executions will be much faster because of stored data in your browser cache.

If you are unfamiliar with the handling of Jupyter notebooks, here are two useful resources:

- [The JupyterLab Interface](https://jupyterlab.readthedocs.io/en/stable/user/interface.html)
- [The Jupyter Notebook](https://jupyterlab.readthedocs.io/en/stable/user/notebook.html)

# Preliminaries

In this part some preliminaries for querying LINDAS with SPARQL are introduced. If you are only interested in the actual LINDAS tutorial, you can skip the whole section and start [here](#Actual-Tutorial).

## Install and Import the Necessary Modules

Querying a SPARQL endpoint is basically making a POST request to the corresponding endpoint URL. As JupyterLite at the moment has no support for Python's `requests` modul, the JavaScript fetch API is used (with some tricks). To make this happen, the following modules have to be importet: 

In [ ]:
%pip install -q folium

In [ ]:
import json
import pandas as pd
import geopandas as gpd
import numpy as np
import folium
import branca
from pyodide.ffi import to_js
from IPython.display import JSON, HTML
from js import Object, fetch
from io import StringIO

## Define Main Query Function

As the JavaScript fetch API is asynchronous, the corresponding Python function `query` has to be declared as `async`. This function allows to query the 3 Swiss governmental triple stores.

In [ ]:
async def query(query_string, store = "L"):
    
    # three Swiss triplestores
    if store == "F":
        address = 'https://fedlex.data.admin.ch/sparqlendpoint'
    elif store == "G":
        address = 'https://geo.ld.admin.ch/query'
    else:
        address = 'https://ld.admin.ch/query'
    
    # try the Post request with help of JS fetch
    # the creation of the request header is a little bit complicated because it needs to be a 
    # JavaScript JSON that is made within a Python source code
    try:
        resp = await fetch(address,
          method="POST",
          body="query=" + query_string,
          credentials="same-origin",
          headers=Object.fromEntries(to_js({"Content-Type": "application/x-www-form-urlencoded; charset=UTF-8", 
                                            "Accept": "text/csv" })),
        )
    except:
        raise RuntimeError("fetch failed")
    
    
    if resp.ok:
        res = await resp.text()
        # ld.admin.ch throws errors starting with '{"message":'
        if '{"message":' in res:
            error = json.loads(res)
            raise RuntimeError("SPARQL query malformed: " + error["message"])
        # geo.ld.admin.ch throws errors starting with 'Parse error:'
        elif 'Parse error:' in res:
            raise RuntimeError("SPARQL query malformed: " + res)
        else:
            # if everything works out, create a pandas dataframe from the csv result
            df = pd.read_csv(StringIO(res))
            return df
    else:
        # fedlex.data.admin.ch throws error with response status 400
        if resp.status == 400:
            raise RuntimeError("Response status 400: Possible malformed SPARQL query. No syntactic advice available.")
        else:
            raise RuntimeError("Response status " + resp.status)

If you are interested in the details of using the JavaScript fetch API within JupyterLite, please consult:

- https://pyodide.org/en/stable/usage/faq.html#how-can-i-use-fetch-with-optional-arguments-from-python
- https://github.com/jupyterlite/jupyterlite/discussions/412
- https://lwebapp.com/en/post/pyodide-fetch

## Define Display Function

Displays pandas dataframe resulting from the SPARQL query as HTML with clickable links.

In [ ]:
def display_result(df):
    df = HTML(df.to_html(render_links=True, escape=False))
    display(df)

# Actual Tutorial

## All Datasets

One of the basic elements in LINDAS are datasets that group similar data. The first task is to query LINDAS for all available datasets. Datasets in LINDAS have the [rdf:type](http://www.w3.org/1999/02/22-rdf-syntax-ns#type) [schema:Dataset](https://schema.org/Dataset).

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>

SELECT * WHERE {
    ?dataset a schema:Dataset;
        schema:name ?name.
        OPTIONAL {
            ?dataset schema:version ?version.
        }
        
    FILTER(lang(?name) = 'de')
}
ORDER BY (?dataset)

""")

display_result(df)

As you can see, there are multiple datasets, that exist in different versions. If you click on e.g. https://environment.ld.admin.ch/foen/ubd000504/3, you will get a useful webpage with additional infos on this dataset. For example, the [purl:creator](http://purl.org/dc/terms/creator) https://register.ld.admin.ch/opendataswiss/org/bundesamt-fur-umwelt-bafu is stated.

## All Named Graphs

In [ ]:
df = await query("""

SELECT ?g 
WHERE {
  GRAPH ?g { }
}

""")

display_result(df)


## Datasets from a Specific Creator

We can use this [purl:creator](http://purl.org/dc/terms/creator) value to refine our query for showing only datasets from the BAFU:

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>

SELECT * WHERE {
    ?dataset a schema:Dataset;
        schema:name ?name;
        purl:creator <https://register.ld.admin.ch/opendataswiss/org/bundesamt-fur-umwelt-bafu>.
        
    FILTER(lang(?name) = 'de')
}

""")

display_result(df)

## Working with a specific Dataset (Bathing water quality)

Assume that we would be intersted in the Bathing water quality dataset. Our first task is to get the latest version of the dataset. If we click on a arbitrary version of the dataset (e.g. https://environment.ld.admin.ch/foen/ubd0104/3/), we see that the datasat has a [purl:identifier](http://purl.org/dc/terms/identifier) that reads "ubd0104". It turns out, that this identifier ist stable through ther versions, so we can query for the latest version with the help of a SPARQL [subquery](https://en.wikibooks.org/wiki/SPARQL/Subqueries) construction:

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>

SELECT * WHERE {

    ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version ?maxversion.

    {
    SELECT (max(?version) as ?maxversion) WHERE {
        ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version ?version.
    }
    }
}

""")

display_result(df)

As this dataset is a [rdf:type](http://www.w3.org/1999/02/22-rdf-syntax-ns#type) [cube:Cube](https://cube.link/#Cube), the actual measurements are organised within a [cube:ObservationSet](https://cube.link/#ObservationSet):

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>

SELECT ?observation WHERE {
    
    ?dataset a schema:Dataset;
        purl:identifier "ubd0104";
        schema:version ?maxversion;
        cube:observationSet/cube:observation ?observation.
    
    {
    SELECT (max(?version) as ?maxversion) WHERE {
        ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version ?version.
    }
    }
}

LIMIT 10

""")

display_result(df)

If we want to have tabular data, we can observe a single observation and see the used vocabulary for creating the following query:

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>
PREFIX ubd0104: <https://environment.ld.admin.ch/foen/ubd0104/>

SELECT ?name ?station ?dateOfProbing ?parameterType ?value WHERE {
    
    ?dataset a schema:Dataset;
        purl:identifier "ubd0104";
        schema:version ?maxversion;
        cube:observationSet/cube:observation ?observation.
    
    ?observation ubd0104:dateofprobing ?dateOfProbing;
        ubd0104:station ?station;
        ubd0104:parametertype ?parameterType;
        ubd0104:station ?station;
        ubd0104:value ?value.
    
    ?station schema:name ?name.
        
    {
    SELECT (max(?version) as ?maxversion) WHERE {
        ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version ?version.
    }
    }
}

LIMIT 100

""")

display_result(df)

Our next goal is to show all the available stations on a map of Switzerland. The SPARQL query is:

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>
PREFIX ubd0104: <https://environment.ld.admin.ch/foen/ubd0104/>


SELECT DISTINCT ?station ?canton ?name ?latitude ?longitude WHERE {
    
    ?dataset a schema:Dataset;
        purl:identifier "ubd0104";
        schema:version ?maxversion;
        cube:observationSet/cube:observation ?observation.
    
    ?observation ubd0104:station ?station.
    ?station ubd0104:kanton ?canton;
        schema:name ?name;
        schema:latitude ?latitude;
        schema:longitude ?longitude.
        
    {
    SELECT (max(?version) as ?maxversion) WHERE {
        ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version ?version.
    }
    }
}

""")

display_result(df)

To display all the stations on a map, we use the Python [folium](https://python-visualization.github.io/folium/) module:

In [ ]:
# create the map and center it around the mean coordinates of the stations
m = folium.Map(location=[df['latitude'].mean(),df['longitude'].mean()], tiles="stamenterrain", zoom_start=8)

# add marker one by one on the map
for i in range(0,len(df)):
    folium.Marker(
        location=[df.iloc[i]['latitude'], df.iloc[i]['longitude']],
        popup=folium.Popup(folium.Html('<a href="' + df.iloc[i]['station'] + '" target="_blank">' + df.iloc[i]['name'] + '</a>', script=True), max_width=500),
        icon=folium.Icon(icon='person-swimming', prefix='fa')
   ).add_to(m)

# show the map
m

To improve the map, we want to display the marker in different colors corresponding to the quality of the last measurements and by clicking on the marker, a popup with additional information should appear.

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>
PREFIX ubd0104: <https://environment.ld.admin.ch/foen/ubd0104/>

SELECT ?name ?station ?latitude ?longitude ?date ?value WHERE {

    ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version 6;
            cube:observationSet/cube:observation ?observation.

        ?observation ubd0104:station ?station;
            ubd0104:parametertype "E.coli";
            ubd0104:dateofprobing ?date;
            ubd0104:value ?value.
        
        ?station schema:name ?name;
            schema:latitude ?latitude;
            schema:longitude ?longitude.
    

    {
    SELECT ?station (max(?date) as ?date) WHERE {

        ?dataset a schema:Dataset;
            purl:identifier "ubd0104";
            schema:version 6;
            cube:observationSet/cube:observation ?observation.

        ?observation ubd0104:station ?station;
            ubd0104:parametertype "E.coli";
            ubd0104:dateofprobing ?date.
    }

    GROUP BY ?station
    }
}
""")

display_result(df)

Now we want to create the map.

In [ ]:
# create column with quality colors
conditions = [(df['value'] < 10),
              (df['value'] >= 10) & (df['value'] < 50),
              (df['value'] >= 50) & (df['value'] < 100),
              (df['value'] >= 100)
             ]

values = ["green", "darkgreen", "orange", "red"]

df['quality'] = np.select(conditions, values)

# create the map and center it around the mean coordinates of the stations
m = folium.Map(location=[df['latitude'].mean(),df['longitude'].mean()], tiles="stamenterrain", zoom_start=8)

# add marker one by one on the map
for i in range(0,len(df)):
    folium.Marker(
        location=[df.iloc[i]['latitude'], df.iloc[i]['longitude']],
        popup=folium.Popup(folium.Html('<p><a href="' + df.iloc[i]['station'] + '" target="_blank">' + df.iloc[i]['name'] + '</a></p>' + 
                                       '<p>Date: ' + df.iloc[i]['date'] + '</p>' + 
                                       '<p>Value E.Coli = ' + str(df.iloc[i]["value"]) + '</p>', script=True), max_width=500),
        icon=folium.Icon(icon='person-swimming', prefix='fa', color = df.iloc[i]['quality'])
   ).add_to(m)

# show the map
m

## Defined Terms

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT * WHERE {
    
    ?DefinedTermSet a schema:DefinedTermSet;
        schema:name ?Name.
    FILTER(regex(str(?DefinedTermSet), "admin.ch" ) )
    FILTER(lang(?Name) = "de")
}

""")

display_result(df)

## geo.ld.admin.ch

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX gn: <http://www.geonames.org/ontology#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?Municipality ?Name ?Population WHERE {
    ?Municipality gn:featureCode gn:A.ADM3 .
    ?Municipality schema:name ?Name .
    ?Municipality gn:population ?Population .
    ?Municipality <http://purl.org/dc/terms/issued> ?Date .
    FILTER (?Date = "2022-01-01"^^xsd:date)
}
ORDER BY DESC(?Population)
LIMIT 5

""", "G")

display_result(df)

## Choropleth Maps

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX gn: <http://www.geonames.org/ontology#>
PREFIX ogis: <http://www.opengis.net/ont/geosparql#>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?bfs ?name ?version ?population ?area ?geometry WHERE {

?muni gn:featureCode gn:A.ADM3;
    schema:name ?name;
    schema:about ?about;
    <https://geo.ld.admin.ch/def/bfsNumber> ?bfs;
    purl:hasVersion ?version.
    
?version purl:issued "2023-01-01"^^xsd:date;
    ogis:defaultGeometry/ogis:asWKT ?geometry;
    gn:population ?population;
    <http://dbpedia.org/property/area> ?area;
    gn:parentADM1 ?canton.
?canton schema:name "Vaud".
    
}

""", "G")

df = gpd.GeoDataFrame(df)

df["geometry"] = gpd.GeoSeries.from_wkt(df["geometry"], crs="EPSG:4326")

df["density"] = df["population"] / df["area"]

sw = df.bounds[["miny", "minx"]].min().values.tolist()
ne = df.bounds[["maxy", "maxx"]].max().values.tolist()

display(df.head())

In [ ]:
bins = list(df["density"].quantile([0, 0.1, 0.9, 1]))

m = folium.Map(location=[47, 7], tiles='openstreetmap', zoom_start=10)
folium.Choropleth(
    geo_data=df,
    data=df,
    name="Population Density",
    columns=["bfs", "density"],
    key_on="feature.properties.bfs",
    fill_color="YlGn",
    fill_opacity=0.7,
    line_opacity=0.2,
    bins=bins,
    legend_name="Population Density [#/HA]",
).add_to(m)
folium.TileLayer('stamenterrain').add_to(m)
folium.TileLayer('stamentoner').add_to(m)
folium.TileLayer('cartodbpositron').add_to(m)
folium.LayerControl().add_to(m)
display(m)

### Choropleth wit GeoJSON

In [ ]:
df["html"] = "<a href='" + df["version"] + "' target='_blank'>" + df["name"] + "</a>"

geo_json = df.to_json()

colorscale = branca.colormap.linear.YlOrRd_09.scale(0, 10)

m = folium.Map(location=[47, 7], tiles='openstreetmap', zoom_start=10)

folium.GeoJson(geo_json, 
               name="geojson",
               style_function = lambda feature: {
                    "fillOpacity": 0.7,
                    "line_opacity": 0.5,
                    "weight": 1,
                    "fillColor": colorscale(feature["properties"]["density"])
               },
               highlight_function=lambda feature: {
                     "fillColor": colorscale(feature["properties"]["density"]),
                     "fillOpacity": 0.2
               },
               tooltip=folium.features.GeoJsonTooltip(
                     fields=["name", "population", "area", "density"],
                     style=("background-color: white; color: #333333; font-family: arial; font-size: 12px; padding: 10px;"),
                     sticky=False
               ),
               popup=folium.features.GeoJsonPopup(
                    fields=["html", "population"],
                    aliases=["Gemeinde", "Einwohner"],
                    localize=False,
                    labels=True
               )
).add_to(m)
folium.LayerControl().add_to(m)
m.fit_bounds([sw, ne])
m

## fedlex.data.admin.ch

In [ ]:
df = await query("""

PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX jolux: <http://data.legilux.public.lu/resource/ontology/jolux#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?eventLabel ?decisionDate ?sourceId  WHERE {
  ?actAbstract jolux:classifiedByTaxonomyEntry ?concept .
  ?concept skos:notation ?notation .
  filter(datatype(?notation) = <https://fedlex.data.admin.ch/vocabulary/notation-type/id-systematique>)
  filter(str(?notation) = '170.512')

  ?actAbstract  jolux:basicAct ?basicAct .
  ?draft jolux:hasResultingLegalResource ?basicAct .
  ?draft rdf:type jolux:InitialDraft .

  ?draft jolux:draftHasLegislativeTask ?event . 
  ?event jolux:legislativeTaskType ?eventType .
  ?eventType skos:prefLabel ?eventLabel . filter(lang(?eventLabel) = 'fr')
  ?event jolux:decisionDate ?decisionDate .
  optional {
    ?event jolux:legislativeTaskHasResultingLegalResource ?source .
    ?source jolux:isRealizedBy ?expression .
    ?expression jolux:historicalLegalId ?sourceId .
    ?expression jolux:language <http://publications.europa.eu/resource/authority/language/FRA> .
  }
} 
order by desc(?decisionDate)

""", "F")

display_result(df)

## Public Transport

### LINDAS: Public Transport Stops

In [ ]:
df = await query("""

SELECT *  
WHERE {
  ?station a <http://vocab.gtfs.org/terms#Station>.
} LIMIT 10

""")

display_result(df)

### LINDAS: Public Transport Stops per Municipality

In [ ]:
df = await query("""

SELECT ?muni (COUNT(?station) as ?number) WHERE {
 
 ?station a <http://vocab.gtfs.org/terms#Station>;
      <https://gont.ch/municipality> ?muni.

} GROUP BY ?muni ORDER BY DESC(?number) LIMIT 10

""")

display_result(df)

### Swisstopo: Train Stations per Municipality

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>

SELECT DISTINCT ?muni_name (COUNT(?stop) as ?number_of_stops) WHERE {
  ?stop a <http://vocab.gtfs.org/terms#Stop>;
      <https://geo.ld.admin.ch/def/transportation/meansOfTransportation> <https://geo.ld.admin.ch/vocabulary/MeansOfTransportation/B>;
      <http://schema.org/containedInPlace> ?muni.
    ?muni schema:name ?muni_name.
} GROUP BY ?muni_name ORDER BY DESC(?number_of_stops) LIMIT 20

""", "G")

display_result(df)

### Swisstopo: Municipalities Withouth any Public Transport Stop

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?muni ?muni_name ?geometry WHERE {

    ?muni a <http://schema.org/AdministrativeArea>;
        <http://www.geonames.org/ontology#featureCode> <http://www.geonames.org/ontology#A.ADM3>;
        <http://purl.org/dc/terms/hasVersion> ?version;
        schema:name ?muni_name.
    ?version <http://purl.org/dc/terms/issued> "2023-01-01"^^xsd:date;
        <http://www.opengis.net/ont/geosparql#defaultGeometry>/<http://www.opengis.net/ont/geosparql#asWKT> ?geometry
        
    FILTER NOT EXISTS {
        ?stop a <http://vocab.gtfs.org/terms#Stop>;
            <http://schema.org/containedInPlace> ?muni.
    }

}

""", "G")

display_result(df[["muni", "muni_name"]])

In [ ]:
df = gpd.GeoDataFrame(df)

df["geometry"] = gpd.GeoSeries.from_wkt(df["geometry"], crs="EPSG:4326")

sw = df.bounds[["miny", "minx"]].min().values.tolist()
ne = df.bounds[["maxy", "maxx"]].max().values.tolist()

display(df.head())

In [ ]:
geo_json = df.to_json()

m = folium.Map(location=[47, 7], tiles='openstreetmap', zoom_start=10)

folium.GeoJson(geo_json, 
               name="geojson",
               style_function = lambda feature: {
                    "fillOpacity": 0.7,
                    "line_opacity": 0.5,
                    "weight": 1,
                    "fillColor": "blue"
               },
               tooltip=folium.features.GeoJsonTooltip(
                     fields=["muni_name"],
                     style=("background-color: white; color: #333333; font-family: arial; font-size: 12px; padding: 10px;"),
                     sticky=False
               )
).add_to(m)
folium.LayerControl().add_to(m)
m.fit_bounds([sw, ne])
m

### Swisstopo: Public Transport Stops with Location in No-More Existent Municipalities

In [ ]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX purl: <http://purl.org/dc/terms/>

SELECT ?stop ?stop_name ?numi_name WHERE {

    ?stop a <http://vocab.gtfs.org/terms#Stop>;
        schema:name ?stop_name;
        schema:containedInPlace ?muni.
    ?muni schema:name ?numi_name.
    
     FILTER NOT EXISTS {
        ?muni purl:hasVersion ?version.    
        ?version purl:issued "2023-01-01"^^xsd:date;
    }
}

""", "G")

display_result(df)